# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama

In [2]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL_GPT = 'gpt-4o-mini'

API key looks good so far


In [3]:
# set up environment
openai = OpenAI()

In [4]:
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_LLAMA = 'llama3.2'

In [5]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

In [6]:
# Get gpt-4o-mini to answer, with streaming
def ask_question_gpt(question, system_prompt):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [7]:
# Get Llama 3.2 to answer
user_prompt = "Please give a detailed explanation to the following question: " + question
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
display(Markdown(reply))

**Code Explanation**

```python
yield from {book.get("author") for book in books if book.get("author")}
```

This line of code is written in Python and utilizes several advanced concepts to generate a sequence of values.

### Breakdown:

- `yield from`: This keyword is used to delegate a sub-expression within a generator function to another generator, allowing the outer generator to yield values from the inner one. It's similar to the `yield` statement, but instead of generating a single value, it yields from multiple sources.

- `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension (due to the curly brackets) that generates a sequence of values.

  - `.get("author")`: Retrieves the "author" key-value pair from each `book` dictionary and returns its value. If the key doesn't exist, it returns `None`.

  - `for book in books`: Iterates over each item in the `books` list (assuming it's a collection of dictionaries).

  - `if book.get("author")`: Filters out any items from the iteration where the "author" key does not exist.

- The entire expression is wrapped in another generator function, which uses `yield from`. This outer function will yield values from each dictionary comprehension.

### How it Works:

1. When you call this generator function, it starts iterating over the `books` list.
2. For each book in the list, it checks if the "author" key exists and retrieves its value using `.get("author")`.
3. If the "author" key does not exist for a particular book, that book is skipped.
4. The generator function yields values from each dictionary comprehension.
5. Since `yield from` delegates to another generator (the dictionary comprehension), it produces a sequence of values where each value corresponds to the author of a book.

### Use Case:

This code could be used in situations where you have a list of books, and you want to extract the authors of only those books that have an "author" key. It's often useful in data processing pipelines or when working with APIs that return dictionaries or objects representing entities.

Example usage:
```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Author C"}
]

def get_book_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}
    
for author in get_book_authors(books):
    print(author)
```
This would output:
```
Author A
Author C
```

In [8]:
# Get gpt-4o-mini to answer, with streaming
ask_question_gpt(question, system_prompt)

The code `yield from {book.get("author") for book in books if book.get("author")}` is a Python expression that is part of a generator function, utilizing the `yield from` syntax. Let's break down what each aspect of the code does and why it is structured this way.

### Breakdown of the Code

1. **Generator Function**:
   - The use of `yield from` suggests that this code is inside a generator function. Generators are a special type of iterator in Python that allow you to iterate through a sequence of values one at a time, without storing the entire sequence in memory.

2. **Set Comprehension**:
   - `{book.get("author") for book in books if book.get("author")}` is a set comprehension. This means it creates a set (a collection of unique items) of authors extracted from a collection of `books`.
   - `book.get("author")` retrieves the value associated with the key `"author"` from each dictionary (representing a book) in the `books` collection. 
   - `if book.get("author")` ensures that only books with a valid (non-None and non-empty) author name are included in the set.

3. **Yielding Authors**:
   - `yield from` is used to yield each item from the generated set of authors to the caller of the generator. This allows the caller to obtain each author one by one, while maintaining the unique constraint of a set.

### Purpose

The purpose of this code is to extract and yield the unique names of authors from a collection of book dictionaries, where each dictionary should contain an `"author"` key. 

### Why Use This Structure?

- **Uniqueness**: Using a set comprehension ensures that each author is only yielded once, even if multiple books are written by the same author.
- **Efficiency**: By using a generator and `yield from`, it allows for efficient memory usage since authors are generated one at a time rather than all at once.
- **Readability**: The combination of set comprehension and the yield syntax makes the code concise and expressive, conveying its intent clearly.

### Example Context

Suppose you have a list of book records where each record is a dictionary containing details about a book including its author. This code would efficiently gather and yield each distinct author's name without duplicates, which can be useful for tasks such as creating an author index or simply listing authors for displaying in a user interface.

In summary, this code serves to generate a unique list of authors from a collection of book dictionaries in an efficient and Pythonic way.